# Instructional Fine Tuining on custom dataset

In [ ]:
from datasets import load_dataset
dataset = load_dataset("Amod/mental_health_counseling_conversations", split="train")

README.md: 0.00B [00:00, ?B/s]

combined_dataset.json: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/3512 [00:00<?, ? examples/s]

In [ ]:
dataset, dataset[0]

(Dataset({
     features: ['Context', 'Response'],
     num_rows: 3512
 }),
 {'Context': "I'm going through some things with my feelings and myself. I barely sleep and I do nothing but think about how I'm worthless and how I shouldn't be here.\n   I've never tried or contemplated suicide. I've always wanted to fix my issues, but I never get around to it.\n   How can I change my feeling of being worthless to everyone?",
  'Response': "If everyone thinks you're worthless, then maybe you need to find new people to hang out with.Seriously, the social context in which a person lives is a big influence in self-esteem.Otherwise, you can go round and round trying to understand why you're not worthless, then go back to the same crowd and be knocked down again.There are many inspirational messages you can find in social media. \xa0Maybe read some of the ones which state that no person is worthless, and that everyone has a good purpose to their life.Also, since our culture is so saturated with th

In [ ]:
def format_row(example):
  question = example["Context"]
  answer = example["Response"]
  example["Text"] = f"[INST] {question} [/INST] {answer}"
  return example

In [ ]:
formatted_dataset = dataset.map(format_row)
formatted_dataset

Map:   0%|          | 0/3512 [00:00<?, ? examples/s]

Dataset({
    features: ['Context', 'Response', 'Text'],
    num_rows: 3512
})

In [ ]:
formatted_dataset, formatted_dataset[0]["Text"]

(Dataset({
     features: ['Context', 'Response', 'Text'],
     num_rows: 3512
 }),
 "[INST] I'm going through some things with my feelings and myself. I barely sleep and I do nothing but think about how I'm worthless and how I shouldn't be here.\n   I've never tried or contemplated suicide. I've always wanted to fix my issues, but I never get around to it.\n   How can I change my feeling of being worthless to everyone? [/INST] If everyone thinks you're worthless, then maybe you need to find new people to hang out with.Seriously, the social context in which a person lives is a big influence in self-esteem.Otherwise, you can go round and round trying to understand why you're not worthless, then go back to the same crowd and be knocked down again.There are many inspirational messages you can find in social media. \xa0Maybe read some of the ones which state that no person is worthless, and that everyone has a good purpose to their life.Also, since our culture is so saturated with the beli

In [ ]:
import pandas as pd

# Convert dataset to dataframe

df=pd.DataFrame(dataset)

In [ ]:
df.to_csv("/content/drive/MyDrive/LLM/Datasets/mental_health_conversational_IFT.csv", index=False)

In [ ]:
df.to_json("/content/drive/MyDrive/LLM/Datasets/mental_health_conversational_IFT.json", orient="records", lines =True)

In [16]:
# Own Dataset loading
from datasets import load_dataset

dataset = load_dataset("csv", data_files="/content/drive/MyDrive/LLM/Datasets/aido_finetune_dataset_v2.csv", split="train")
dataset

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['instruction', 'Question', 'Response'],
    num_rows: 20
})

In [17]:
def format_example(example):
  prompt = f"### Instruction:\n{example['instruction']}\n\n### Input:\n{example['Question']}\n### Response:\n{example['Question']}"
  return {"text":prompt}

In [18]:
dataset = dataset.map(format_example)

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

In [19]:
dataset

Dataset({
    features: ['instruction', 'Question', 'Response', 'text'],
    num_rows: 20
})

In [20]:
dataset[0]

{'instruction': 'Define the following term in the context of biology and AI.',
 'Question': 'What is an AI-Driven Digital Organism (AIDO)?',
 'Response': 'An AI-Driven Digital Organism (AIDO) is an integrated multiscale system of foundation models built specifically for biology. It combines multiple component models spanning molecular, cellular, and organismal levels to enable prediction, simulation, and reprogramming of biological activities in a unified and holistic manner.',
 'text': '### Instruction:\nDefine the following term in the context of biology and AI.\n\n### Input:\nWhat is an AI-Driven Digital Organism (AIDO)?\n### Response:\nWhat is an AI-Driven Digital Organism (AIDO)?'}

In [21]:
dataset['text'][0]

'### Instruction:\nDefine the following term in the context of biology and AI.\n\n### Input:\nWhat is an AI-Driven Digital Organism (AIDO)?\n### Response:\nWhat is an AI-Driven Digital Organism (AIDO)?'

In [25]:
from  transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments, DataCollatorForLanguageModeling, BitsAndBytesConfig

from datasets import load_dataset

In [35]:
non_instruction_model_path = "/content/drive/MyDrive/LLM/LLM_NFT_QLORA/checkpoint-8"

In [36]:
non_instruction_model =AutoModelForCausalLM.from_pretrained(non_instruction_model_path)

Loading weights:   0%|          | 0/201 [00:01<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/peft/config.py:220: UserWarning: Unexpected keyword arguments ['lora_ga_config', 'use_bdlora'] for class LoraConfig, these are ignored. This probably means that you're loading a configuration file that was saved using a higher version of the library and additional parameters have been introduced since. It is highly recommended to upgrade the PEFT version before continuing (e.g. by running `pip install -U peft`).
  warnings.warn(


Loading weights:   0%|          | 0/88 [00:00<?, ?it/s]

In [38]:
print(non_instruction_model)

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(32000, 2048)
    (layers): ModuleList(
      (0-21): 22 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): lora.Linear(
            (base_layer): Linear(in_features=2048, out_features=2048, bias=False)
            (lora_dropout): ModuleDict(
              (default): Dropout(p=0.05, inplace=False)
            )
            (lora_A): ModuleDict(
              (default): Linear(in_features=2048, out_features=8, bias=False)
            )
            (lora_B): ModuleDict(
              (default): Linear(in_features=8, out_features=2048, bias=False)
            )
            (lora_embedding_A): ParameterDict()
            (lora_embedding_B): ParameterDict()
            (lora_magnitude_vector): ModuleDict()
          )
          (k_proj): Linear(in_features=2048, out_features=256, bias=False)
          (v_proj): lora.Linear(
            (base_layer): Linear(in_features=2048, out_features=256, bia

In [39]:
print(non_instruction_model.config)

LlamaConfig {
  "architectures": [
    "LlamaForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 1,
  "dtype": "float32",
  "eos_token_id": 2,
  "head_dim": 64,
  "hidden_act": "silu",
  "hidden_size": 2048,
  "initializer_range": 0.02,
  "intermediate_size": 5632,
  "max_position_embeddings": 2048,
  "mlp_bias": false,
  "model_type": "llama",
  "num_attention_heads": 32,
  "num_hidden_layers": 22,
  "num_key_value_heads": 4,
  "pad_token_id": null,
  "pretraining_tp": 1,
  "rms_norm_eps": 1e-05,
  "rope_parameters": {
    "rope_theta": 10000.0,
    "rope_type": "default"
  },
  "tie_word_embeddings": false,
  "transformers_version": "5.0.0",
  "use_cache": true,
  "vocab_size": 32000
}



In [40]:
total_params = sum(p.numel() for p in non_instruction_model.parameters())
trainable_params = sum(p.numel() for p in non_instruction_model.parameters() if p.requires_grad)

print(f"Total params: {total_params:,}")
print(f"Trainable params: {trainable_params:,}")

Total params: 1,101,174,784
Trainable params: 1,126,400


In [41]:
print("Context length:", non_instruction_model.config.max_position_embeddings)

Context length: 2048


In [42]:
print(non_instruction_model.config.model_type)

llama


In [44]:
pip install torchinfo


In [45]:
from torchinfo import summary

summary(non_instruction_model)

Layer (type:depth-idx)                                  Param #
LlamaForCausalLM                                        --
├─LlamaModel: 1-1                                       --
│    └─Embedding: 2-1                                   (65,536,000)
│    └─ModuleList: 2-2                                  --
│    │    └─LlamaDecoderLayer: 3-1                      44,095,488
│    │    └─LlamaDecoderLayer: 3-2                      44,095,488
│    │    └─LlamaDecoderLayer: 3-3                      44,095,488
│    │    └─LlamaDecoderLayer: 3-4                      44,095,488
│    │    └─LlamaDecoderLayer: 3-5                      44,095,488
│    │    └─LlamaDecoderLayer: 3-6                      44,095,488
│    │    └─LlamaDecoderLayer: 3-7                      44,095,488
│    │    └─LlamaDecoderLayer: 3-8                      44,095,488
│    │    └─LlamaDecoderLayer: 3-9                      44,095,488
│    │    └─LlamaDecoderLayer: 3-10                     44,095,488
│    │    └─LlamaDec

In [27]:
saved_path = "/content/drive/MyDrive/LLM/TinyLlama-1.1B-intermediate-step-1431k-3T"

In [28]:
tokenizer = AutoTokenizer.from_pretrained(saved_path)

In [37]:
print(tokenizer)
print("Vocab size:", tokenizer.vocab_size)
print("Model max length:", tokenizer.model_max_length)

TokenizersBackend(name_or_path='/content/drive/MyDrive/LLM/TinyLlama-1.1B-intermediate-step-1431k-3T', vocab_size=32000, model_max_length=1000000000000000019884624838656, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '<unk>', 'pad_token': '</s>'}, added_tokens_decoder={
	0: AddedToken("<unk>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("<s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
)
Vocab size: 32000
Model max length: 1000000000000000019884624838656


In [29]:
if tokenizer.pad_token is None:
  tokenizer.pad_token = tokenizer.eos_token


In [30]:
def tokenize_fn(example):
  tokens = tokenizer(example["text"], truncation=True, padding="max_length", max_length=512)
  tokens["labels"] = tokens["input_ids"].copy()
  return tokens

In [ ]:
# Tokenize with response masking (Second method)

def tokenize_and_mask(example):
  text=example['text']

  # Tokenized full text
  enc = tokenizer(text, truncation=True, padding="max_length", max_length=512)
  input_ids = enc["input_ids"]

  # Find where '### Response:' starts

  response_marker = '### Response:'
  response_start = text.find(response_marker)

  if response_start != -1:
    # Token index where response beigns
    response_token_start=len(tokenizer(text[:response_start])["input_ids"])

  else:
    response_token_start = 0 # if marher is not found

  #Clone labels and mask out evrything before 'Response'
  labels = enc["input_ids"].copy()
  labels[:response_token_start] = -100 * response_token_start

  enc['labels'] = labels
  return enc



In [31]:
tokenized = dataset.map(tokenize_fn, batched=True)

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

In [33]:
tokenized[0]

{'instruction': 'Define the following term in the context of biology and AI.',
 'Question': 'What is an AI-Driven Digital Organism (AIDO)?',
 'Response': 'An AI-Driven Digital Organism (AIDO) is an integrated multiscale system of foundation models built specifically for biology. It combines multiple component models spanning molecular, cellular, and organismal levels to enable prediction, simulation, and reprogramming of biological activities in a unified and holistic manner.',
 'text': '### Instruction:\nDefine the following term in the context of biology and AI.\n\n### Input:\nWhat is an AI-Driven Digital Organism (AIDO)?\n### Response:\nWhat is an AI-Driven Digital Organism (AIDO)?',
 'input_ids': [1,
  835,
  2799,
  4080,
  29901,
  13,
  3206,
  457,
  278,
  1494,
  1840,
  297,
  278,
  3030,
  310,
  4768,
  3002,
  322,
  319,
  29902,
  29889,
  13,
  13,
  2277,
  29937,
  10567,
  29901,
  13,
  5618,
  338,
  385,
  319,
  29902,
  29899,
  29928,
  374,
  854,
  15918,
 

In [34]:
from peft import LoraConfig, get_peft_model, TaskType

In [48]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj", "v_proj"]
)

In [50]:
model = get_peft_model(non_instruction_model, lora_config)

In [51]:
args =TrainingArguments(
    output_dir="/content/drive/MyDrive/LLM/LLM_IFT_QLORA",
    num_train_epochs=4,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=20,
    save_total_limit=1,
    report_to="none"
)

In [52]:
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized,
)

In [53]:
trainer.train()

Step,Training Loss


TrainOutput(global_step=12, training_loss=12.89694341023763, metrics={'train_runtime': 23.7101, 'train_samples_per_second': 3.374, 'train_steps_per_second': 0.506, 'total_flos': 254518587555840.0, 'train_loss': 12.89694341023763, 'epoch': 4.0})

In [54]:
instruction_model_path = "/content/drive/MyDrive/LLM/LLM_IFT_QLORA/checkpoint-12"

In [61]:
instruction_model= AutoModelForCausalLM.from_pretrained(instruction_model_path, device_map="auto")


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/88 [00:00<?, ?it/s]

In [56]:
prompt="What are the three phases proposed for constructing an AIDO?"

In [57]:
inputs =tokenizer(prompt, return_tensors="pt").to("cuda")

In [64]:
outputs = instruction_model.generate(
    **inputs,
    max_new_tokens=1024,
    temperature=0.2,
    top_p=0.9,
    do_sample=True,
    repetition_penalty=1.1
)

In [65]:
print("\n model Output: \n")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))


 model Output: 

What are the three phases proposed for constructing an AIDO?
A. The first phase is to develop a conceptual design of the AIDO, which will be followed by a detailed design and then construction.
B. The first phase is to develop a conceptual design of the AIDO, which will be followed by a detailed design and then construction.
C. The first phase is to develop a conceptual design of the AIDO, which will be followed by a detailed design and then construction.
D. The first phase is to develop a conceptual design of the AIDO, which will be followed by a detailed design and then construction.
E. The first phase is to develop a conceptual design of the AIDO, which will be followed by a detailed design and then construction.
49. Which of the following is not a characteristic of the AIDO?
A. It is a high-speed train that can travel at speeds up to 300 km/h.
B. It has a maximum speed of 250 km/h.
C. It has a maximum speed of 160 km/h.
D. It has a maximum speed of 180 km/h.
E. It